<a href="https://colab.research.google.com/github/IvaBoneva/V06_project/blob/main/06_V06_%D0%98%D0%B2%D0%B0_%D0%9A%D0%B0%D0%BC%D0%B5%D0%BD%D0%BE%D0%B2%D0%B0_%D0%91%D0%BE%D0%BD%D0%B5%D0%B2%D0%B0_%D1%88%D0%B0%D0%B1%D0%BB%D0%BE%D0%BD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Курсов проект по дисциплината
„Планиране на експеримента“

## Ива Каменова Бонева - вариант V06

**Фак. № 471223035 | Група 76**

Това е Вашето индивидуално задание. Работите само върху файловете на варианта, разпределен по реда на списъка. Задачата е интегрирана и включва последователни етапи от всички 7 теми от конспекта: почистване на данни, разпределения, корелация, извадки, моделиране, прогнозиране и оптимизация.

## 1. Данни за Вашия вариант

Код на варианта: V06

Контекст: Индустриална пещ - температурен датчик - подвариант F

Фактор z_t: Външна температура

Файлове за работа: student_data/V06_raw.csv и student_data/V06_future.csv

Структура на входните файлове: Vxx_raw.csv съдържа t, x_norm, z_factor, y_raw; Vxx_future.csv съдържа t, x_norm, z_factor.

Размер и особености на данните: 120 наблюдения за t = 1..120, хоризонт за прогноза t = 121..126, 4 липсващи стойности и 3 аномални точки в наблюдавания сигнал.

Важно ограничение: използвайте само student_data файловете за Вашия вариант. Не използвайте instructor_data, truth файлове или предварително известни параметри на генератора.

Специален акцент за Вашия вариант: В етап 1 сравнете поне филтрите F(1,1) и F(2,2) и аргументирайте избора на по-консервативен CLEAN pipeline.

## 2. Общата задача

Да се изгради напълно възпроизводим програмен проект, който започва от RAW данни, преминава през първична обработка, статистически анализ, моделиране и прогноза, и завършва с оптимизация на параметрите на избрания модел. Всички етапи трябва да бъдат реализирани с код; не се допуска решение, основано само на ръчна обработка в Excel или LibreOffice.

## Как да използвате този шаблон

Попълвайте TODO секциите с Вашия код и кратки пояснения. Не изтривайте ключовите клетки за импорти, зареждане на данните и финални проверки. Добра практика е след всяка по-съществена стъпка да добавяте кратки интерпретации в Markdown клетка.

In [ ]:
# Основни библиотеки
import sys
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (10, 4)

print("Python:", sys.version.split()[0])
print("NumPy :", np.__version__)
print("Pandas:", pd.__version__)


Python: 3.12.13
NumPy : 2.0.2
Pandas: 2.2.2


In [ ]:
# Помощни функции
def find_file(filename: str) -> Path:
    search_roots = [Path.cwd(), Path.cwd().parent]
    candidates = []
    for root in search_roots:
        candidates.extend([
            root / "student_data" / filename,
            root / filename,
            root / "data" / filename,
        ])
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Не откривам файла: {filename}. Проверете структурата на проекта.")


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs(y_true - y_pred))


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def bias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(y_pred - y_true)


def empirical_edf(x):
    x = np.sort(np.asarray(x, dtype=float))
    p = np.arange(1, len(x) + 1) / len(x)
    return x, p


def ols_fit(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    y_hat = X @ beta
    residuals = y - y_hat
    return beta, y_hat, residuals


def standardize_from_train(train_series, full_series):
    mu = float(np.mean(train_series))
    sd = float(np.std(train_series, ddof=1))
    if sd == 0:
        raise ValueError("Стандартното отклонение е 0. Проверете регресора.")
    return (full_series - mu) / sd, mu, sd


def project_box(theta, lower, upper):
    theta = np.asarray(theta, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    return np.minimum(np.maximum(theta, lower), upper)


def fit_ar1_ols(residuals):
    r = np.asarray(residuals, dtype=float)
    r_prev = r[:-1]
    r_next = r[1:]
    phi = np.dot(r_prev, r_next) / np.dot(r_prev, r_prev)
    eta = r_next - phi * r_prev
    sigma_eta = np.std(eta, ddof=1)
    return phi, sigma_eta


In [ ]:
ASSIGNMENT = {
    "student_name": 'Ива Каменова Бонева',
    "faculty_no": '471223035',
    "group": '76',
    "variant_code": 'V06',
    "scenario_title": 'Индустриална пещ - температурен датчик - подвариант F',
    "factor_label": 'Външна температура',
    "raw_file": 'V06_raw.csv',
    "future_file": 'V06_future.csv',
}

PROJECT_ROOT = Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
SOURCE_DIR = PROJECT_ROOT / "source"
RESULTS_DIR.mkdir(exist_ok=True)
SOURCE_DIR.mkdir(exist_ok=True)

RAW_FILE = find_file(ASSIGNMENT["raw_file"])
FUTURE_FILE = find_file(ASSIGNMENT["future_file"])

raw_df = pd.read_csv(RAW_FILE)
future_df = pd.read_csv(FUTURE_FILE)

display(pd.DataFrame([ASSIGNMENT]))
display(raw_df.head())
display(future_df.head())

print("RAW shape   :", raw_df.shape)
print("FUTURE shape:", future_df.shape)


FileNotFoundError: Не откривам файла: V06_raw.csv. Проверете структурата на проекта.

## Етап 0. Подготовка на проекта

Задание: Създайте проектова структура, импортирайте данните, проверете типовете на колоните, установете броя на редовете и запазете непроменено копие на RAW данните.

Документирайте с каква команда се стартира проектът.

Опишете използваните библиотеки и версията на езика/средата в README.txt.

Подгответе папки поне source/ и results/ за кода и резултатите.

In [ ]:
# TODO: Потвърдете структурата и типовете на входните данни.
display(raw_df.dtypes.to_frame("dtype"))
display(future_df.dtypes.to_frame("dtype"))

# TODO: Създайте непроменено копие на RAW данните.
raw_original = raw_df.copy(deep=True)

# TODO: Добавете проверки за:
# 1) липсващи стойности по колони
# 2) дублирани редове
# 3) монотонност на t
# 4) очакван брой редове
# 5) диапазони на x_norm и z_factor

print("Липсващи стойности в RAW:")
display(raw_df.isna().sum().to_frame("count"))
print("Дублирани редове:", raw_df.duplicated().sum())
print("t е монотонно нарастваща колона:", raw_df["t"].is_monotonic_increasing)

# TODO: Опишете в README.txt как се стартира проектът и каква е структурата на папките.


## Етап 1. Първична обработка на данни

Задание: Постройте графика y_raw(t), открийте липсващите стойности и аномалните точки, сравнете поне един локален филтър от лекцията и още един допълнителен критерий по избор, интерполирайте липсващите стойности и постройте изгладен ред y_smooth(t).

Сравнете поне два локални подхода, например F(1,1), F(2,1) или F(2,2), и аргументирайте избора на финален CLEAN pipeline.

Създайте CLEAN.csv с най-малко следните колони: t, x_norm, z_factor, y_raw, y_clean, is_missing, is_outlier. Препоръчително е да добавите и y_smooth.

Покажете графика с поне raw, y_clean и y_smooth върху една и съща времева ос.

In [ ]:
# Работно копие
df = raw_df.copy()

# TODO 1: маркирайте липсващите стойности
# df["is_missing"] = df["y_raw"].isna()

# TODO 2: реализирайте критерий(и) за аномални точки.
# Примерни идеи:
# - локални разлики / втори разлики
# - rolling median / rolling MAD
# - сравнение на F(1,1), F(2,1), F(2,2)

# TODO 3: интерполирайте липсващите стойности -> y_interp

# TODO 4: коригирайте или филтрирайте outlier-и -> y_clean

# TODO 5: приложете изглаждане -> y_smooth

# TODO 6: създайте CLEAN.csv с колоните:
# t, x_norm, z_factor, y_raw, y_clean, is_missing, is_outlier
# Препоръчително: y_smooth

# TODO 7: визуализация на raw / clean / smooth
# plt.figure()
# plt.plot(df["t"], df["y_raw"], label="y_raw")
# plt.plot(df["t"], df["y_clean"], label="y_clean")
# plt.plot(df["t"], df["y_smooth"], label="y_smooth")
# plt.legend()
# plt.show()


## Етап 2. Случайни величини и разпределения

Задание: Дефинирайте случаен компонент e_t = y_clean(t) - y_smooth(t). Оценете средна стойност, дисперсия, стандартно отклонение, медиана, квартилите и коефициент на вариация.

Постройте хистограма и емпирична функция на разпределение за e_t.

Сравнете e_t с нормално разпределение N(m_hat, s_hat^2). Не е нужно формален тест, но е нужен аргументиран коментар.

Изведете кратък извод дали допускането за нормалност е разумно за Вашите данни.

In [ ]:
# TODO: уверете се, че df съдържа y_clean и y_smooth
# e_t = df["y_clean"] - df["y_smooth"]

# TODO: изчислете mean, var, std, median, Q1, Q3, CV
# stats_e = { ... }

# TODO: построете histogram и empirical EDF за e_t
# x_edf, p_edf = empirical_edf(e_t)

# TODO: сравнете визуално e_t с N(m_hat, s_hat^2) и направете кратък извод


## Етап 3. Двумерни случайни величини, ковариация и корелация

Задание: Разгледайте двойките (z_t, y_clean(t)) за t = 1..120. Изчислете извадкова ковариация и коефициент на корелация на Пирсън.

Постройте scatter plot на z_t спрямо y_clean.

Коментирайте дали зависимостта изглежда линейна, силна, слаба или потенциално нелинейна.

Не се ограничавайте само до числената стойност на r; тълкувайте резултата в контекста на варианта.

In [ ]:
# TODO: използвайте двойките (z_t, y_clean(t))
# z = df["z_factor"].to_numpy()
# y = df["y_clean"].to_numpy()

# TODO: изчислете sample covariance и Pearson correlation
# cov_zy = ...
# corr_zy = ...

# TODO: scatter plot и коментар за линейност / нелинейност
# plt.figure()
# plt.scatter(df["z_factor"], df["y_clean"], s=20)
# plt.xlabel("z_factor")
# plt.ylabel("y_clean")
# plt.show()


## Етап 4. Данни от извадка - групирани и негрупирани оценки

Задание: Третирайте y_clean като извадка от генерална съвкупност. Изчислете статистиките първо върху негрупирани данни, а след това върху групирани данни с 8 равни интервала.

Сравнете средна стойност, медиана, дисперсия, IQR и коефициент на вариация за двата подхода.

Покажете таблица grouped срещу ungrouped.

Коментирайте каква информация се губи при групиране на извадката.

In [ ]:
# TODO: негрупирани статистики за y_clean
# ungrouped = { ... }

# TODO: групирайте y_clean в 8 равни интервала
# bins = np.linspace(df["y_clean"].min(), df["y_clean"].max(), 9)
# df["bin"] = pd.cut(df["y_clean"], bins=bins, include_lowest=True)

# TODO: оценете grouped статистики
# grouped = { ... }

# TODO: създайте сравнителна таблица grouped vs ungrouped
# comparison_df = pd.DataFrame(...)
# display(comparison_df)


## Етап 5. Моделиране и оценка на грешката

Задание: Използвайте разделяне train/test с train = t = 1..90 и test = t = 91..120. Стандартизирайте регресорите по train частта:

u_t = (x_norm - mean_train(x_norm)) / std_train(x_norm),
v_t = (z_factor - mean_train(z_factor)) / std_train(z_factor)

Напаснете с МНМК следните два модела:

M1: y_t = beta0 + beta1*u_t + beta2*v_t + eps_t
M2: y_t = beta0 + beta1*u_t + beta2*v_t + beta3*v_t^2 + eps_t

Изчислете MAE, RMSE и Bias върху train и test.

Изберете по-добрия модел и защитете избора си с числа и кратък коментар.

Покажете поне една графика на напасването и една таблица с оценените параметри.

In [ ]:
# Подгответе CLEAN данните за моделиране
clean_df = df.copy()

train_df = clean_df[clean_df["t"] <= 90].copy()
test_df = clean_df[(clean_df["t"] >= 91) & (clean_df["t"] <= 120)].copy()

# TODO: стандартизация само по train частта
# clean_df["u_t"], x_mean, x_std = standardize_from_train(train_df["x_norm"], clean_df["x_norm"])
# clean_df["v_t"], z_mean, z_std = standardize_from_train(train_df["z_factor"], clean_df["z_factor"])

# TODO: обновете train_df и test_df след добавяне на u_t и v_t

# TODO: формирайте дизайн матриците за M1 и M2
# M1: beta0 + beta1*u_t + beta2*v_t
# M2: beta0 + beta1*u_t + beta2*v_t + beta3*v_t^2

# Пример:
# X_train_m1 = np.column_stack([np.ones(len(train_df)), train_df["u_t"], train_df["v_t"]])
# y_train = train_df["y_clean"].to_numpy()
# beta_m1, yhat_train_m1, resid_train_m1 = ols_fit(X_train_m1, y_train)

# TODO: изчислете MAE, RMSE и Bias върху train и test
# TODO: изберете по-добрия модел и аргументирайте избора
# TODO: покажете графика на fit и таблица с параметрите


## Етап 6. Стохастично прогнозиране

Задание: За избрания модел дефинирайте остатъци r_t = y_clean(t) - y_hat(t) върху train частта. Оценете AR(1) модел r_t = phi*r_(t-1) + eta_t с метод на най-малките квадрати и използвайте V06_future.csv за прогноза на y_t за t = 121..126.

Представете точкова прогноза и приблизителен 95% прогнозен интервал.

Създайте forecast.csv с най-малко колони: t, x_norm, z_factor, y_forecast, lower95, upper95.

Покажете графика на прогнозата заедно с последната част на наблюдавания ред.

In [ ]:
# TODO: изчислете остатъците върху train за избрания модел
# residuals_train = ...

# TODO: оценете AR(1) модел върху residuals_train
# phi, sigma_eta = fit_ar1_ols(residuals_train)

# TODO: стандартизирайте future_df с train mean/std
# future_df["u_t"] = ...
# future_df["v_t"] = ...

# TODO: генерирайте точкова прогноза за t = 121..126
# TODO: добавете приблизителен 95% прогнозен интервал

# TODO: създайте forecast_df с колоните:
# t, x_norm, z_factor, y_forecast, lower95, upper95

# TODO: запишете forecast.csv
# forecast_df.to_csv("forecast.csv", index=False)


## Етап 7. Градиентни / квазиградиентни методи

Задание: Преоценете параметрите на избрания модел чрез projected gradient върху критерия J(theta) = mean((y - y_hat)^2) в train частта.

Използвайте стартова точка theta^(0) = (mean(y_train), 0, 0, 0).

Наложете ограничения: beta0 in [mean(y_train)-30, mean(y_train)+30], beta1, beta2, beta3 in [-20, 20].

Използвайте стъпка rho_k = 1/(k+1). Стоп критерий: ||theta^(k+1)-theta^(k)||_2 < 1e-6 или k = 5000.

Покажете графика на J(theta_k) спрямо k и сравнете получения резултат с МНМК. Ако е избран M1, игнорирайте beta3.

In [ ]:
# TODO: реализирайте projected gradient за избрания модел
# Идея:
# 1) дефинирайте J(theta) = mean((y - X @ theta)^2)
# 2) дефинирайте аналитичен градиент grad J(theta)
# 3) прилагайте проекция след всяка стъпка
# 4) пазете history = [J(theta_k)]

# Примерен шаблон:
# def mse_objective(theta, X, y):
#     y_hat = X @ theta
#     return np.mean((y - y_hat) ** 2)
#
# def mse_gradient(theta, X, y):
#     residual = X @ theta - y
#     return (2 / len(y)) * (X.T @ residual)
#
# def projected_gradient(X, y, theta0, lower, upper, max_iter=5000, tol=1e-6):
#     theta = np.asarray(theta0, dtype=float).copy()
#     history = []
#     for k in range(max_iter):
#         rho_k = 1.0 / (k + 1)
#         grad = mse_gradient(theta, X, y)
#         theta_new = project_box(theta - rho_k * grad, lower, upper)
#         history.append(mse_objective(theta_new, X, y))
#         if np.linalg.norm(theta_new - theta) < tol:
#             theta = theta_new
#             break
#         theta = theta_new
#     return theta, history

# TODO: сравнете параметрите и грешката с МНМК
# TODO: покажете графика J(theta_k) спрямо k


## 4. Задължителни изходни файлове и материали

Архивът за предаване трябва да се казва 471223035_V06_PE.zip.

Архивът трябва да съдържа поне: report.pdf, README.txt, source/, results/, CLEAN.csv и forecast.csv.

report.pdf трябва да е 8-15 страници без приложения и да съдържа кратка теория, метод, резултати и изводи.

README.txt трябва да съдържа команда за стартиране и кратко описание на файловете.

results/ трябва да съдържа графики и таблици, използвани в отчета.

## 5. Минимален набор графики и таблици

raw графика на y_raw(t)

cleaned vs smooth графика

хистограма и EDF на e_t

scatter plot на z_t срещу y_clean

таблица grouped срещу ungrouped статистики

графика или таблица за model fit и грешки

графика на прогнозата с 95% интервал

графика на сходимостта на projected gradient

## 6. Програмни изисквания

Решението трябва да е напълно програмно и възпроизводимо.

Допускат се Python, MATLAB, R, Julia, C/C++ или друг език, но целият pipeline трябва да може да се изпълни с код.

Не се допуска ръчна обработка като основен метод на решението.

Всички формули, допускания и избрани критерии трябва да бъдат ясно описани в отчета.

## 7. Контролен checklist преди предаване

- [ ] Използвах само файловете за моя вариант.
- [ ] Имам CLEAN.csv и forecast.csv с коректни колони.
- [ ] Сравних M1 и M2 и избрах модел по обоснован начин.
- [ ] Направих прогноза за t = 121..126 и добавих интервал.
- [ ] Изпълних optimization етапа и показах сходимост.
- [ ] README.txt съдържа команда за стартиране.
- [ ] report.pdf съдържа всички графики, таблици и изводи.

In [ ]:
# Финални експорти и бързи проверки
# TODO: разкоментирайте, след като сте подготвили крайните таблици
# df.to_csv("CLEAN.csv", index=False)
# forecast_df.to_csv("forecast.csv", index=False)

# TODO: проверка на наличието на задължителните файлове
required = [Path("CLEAN.csv"), Path("forecast.csv")]
for p in required:
    print(f"{p.name}: {'OK' if p.exists() else 'MISSING'}")


---

**Бележка:** Този notebook е шаблон за Ива Каменова Бонева (V06). Попълнете го с Вашето решение, а при предаване включете и отчет, README и изходните CSV файлове.